In [5]:
import pandas as pd
df = pd.read_csv("data/sample_transactions.csv")

df["date"] = pd.to_datetime(df["date"])
df["date"].dtype

df["date"].dt.month

pd.to_datetime(df["date"], errors="coerce")

test = pd.Series(["2026-01-01", "not a date"])
pd.to_datetime(test, errors="coerce")

test = pd.DataFrame({
    "date": ["2026-01-01", "not a date", "2026-01-03"],
    "amount": [100, 200, "abc"]
})

test["date"] = pd.to_datetime(test["date"], errors="coerce")
test["amount"] = pd.to_numeric(test["amount"], errors="coerce")
test

test["date"].isna()

bad = test["date"].isna() | test["amount"].isna()
bad

test[bad]     # the rejected rows
test[~bad]    # the good rows

test2 = pd.DataFrame({
    "description": ["Swiggy", "  ", None, "Netflix"],
    "amount": [-400, -100, -50, -649]
})

test2["description"] = test2["description"].str.strip()
test2

missing = test2["description"].isna() | (test2["description"] == "")
missing

test2.loc[missing, "description"] = "(no description)"
test2

test3 = pd.DataFrame({
    "date": ["2026-09-12", "2026-09-12", "2026-09-13"],
    "description": ["Swiggy Order", "Swiggy Order", "Swiggy Order"],
    "amount": [-450, -450, -450]
})

test3.duplicated(subset=["date", "description", "amount"])

is_dup = test3.duplicated(subset=["date", "description", "amount"])
test3[~is_dup]

is_dup.sum()



np.int64(1)

In [2]:
import os
os.chdir(r"C:\Users\Hari Prasath\moneymap")


In [7]:
from src.data_loader import load_transactions
from src.data_cleaning import clean_transactions

raw = load_transactions("data/sample_transactions.csv")
clean, rejected, summary = clean_transactions(raw)
summary


rules = {
    "swiggy": "Food Delivery",
    "zomato": "Food Delivery",
    "netflix": "Subscriptions",
}

desc = "Random Shop"

for keyword, category in rules.items():
    if keyword in desc.lower():
        print(category)

In [16]:
def categorize(desc):
    for keyword, category in rules.items():
        if keyword in desc.lower():
            return category
    return "Uncategorized"

categorize("Swiggy Order")
categorize("Zomato Order")
categorize("random shop")

clean["category"] = clean["description"].apply(categorize)
clean.head(10)

clean["category"].value_counts()

rules = {
    "refund": "Refunds",
    "credit card payment": "Credit Card Payment",
    "own savings": "Internal Transfer",
    "salary": "Salary",
    "freelance": "Freelance Income",
    "rent": "Housing",
    "swiggy": "Food Delivery",
    "zomato": "Food Delivery",
    "netflix": "Subscriptions",
    "spotify": "Subscriptions",
    "groceries": "Groceries",
    "uber": "Transportation",
    "petrol": "Fuel",
    "electricity": "Utilities",
    "amazon": "Shopping",
    "myntra": "Shopping",
    "udemy": "Education",
    "pvr": "Entertainment",
    "pharmacy": "Healthcare",
    "clinic": "Healthcare",
    "indigo": "Travel",
}

clean["category"] = clean["description"].apply(categorize)
clean.head(10)

clean["category"].value_counts()

income = clean[clean["amount"] > 0]["amount"].sum()
expenses = clean[clean["amount"] < 0]["amount"].sum()
income, expenses

(income + expenses) / income * 100

np.float64(18.357244318181817)

In [22]:
transfers = clean["category"] == "Internal Transfer"
refunds = clean["category"] == "Refunds"

# leave out transfers and refunds
money = clean[~transfers & ~refunds]

income = money[money["amount"] > 0]["amount"].sum()
expenses = money[money["amount"] < 0]["amount"].sum()

# refunds reduce expenses
expenses = expenses + clean[refunds]["amount"].sum()

income, expenses

(income + expenses) / income * 100

cc = clean["category"] == "Credit Card Payment"
cc_total = clean[cc]["amount"].sum()

expenses_without_cc = expenses - cc_total   # minus a negative = adds back
(income + expenses_without_cc) / income * 100

from src.analytics import calculate_summary

calculate_summary(clean)

calculate_summary(clean, count_card_payments=False)

{'income': np.int64(140000),
 'expenses': np.int64(77653),
 'net_cash_flow': np.int64(62347),
 'savings_rate': np.float64(44.53357142857143)}

In [4]:
import os
os.chdir(r"C:\Users\Hari Prasath\moneymap")
from src.data_loader import load_transactions
from src.data_cleaning import clean_transactions
from src.categorization import add_categories
from src.analytics import calculate_summary

raw = load_transactions("data/sample_transactions.csv")
clean, rejected, summary = clean_transactions(raw)
clean = add_categories(clean)

calculate_summary(clean)

spend = clean[(clean["amount"] < 0) & (clean["category"] != "Internal Transfer")]

by_cat = spend.groupby("category")["amount"].sum().abs().sort_values(ascending=False)
by_cat

category
Housing                36000
Credit Card Payment    21500
Groceries               9850
Shopping                8250
Food Delivery           5180
Travel                  4800
Fuel                    4650
Utilities               3830
Subscriptions           2304
Healthcare              1550
Transportation           840
Entertainment            700
Education                499
Name: amount, dtype: int64